# NetCDF data visualization with Xarray

This tutorial is a modified and simplified version of the [Xarray tutorials](https://xarray-contrib.github.io/xarray-tutorial/index.html). The tutorials are freely available for download on [GitHub](https://github.com/xarray-contrib/xarray-tutorial). Some material has also been taken from the [Geohack week](https://geohackweek.github.io/nDarrays/01-introduction/)

## Learning Objectives

- Extracting data from a NetCDF file using xarray
- Indexing and subsetting for visualization
- Integration with cartopy
- Masking


In [ ]:
import xarray as xr
# we will use these modules later
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

In [ ]:
# Load the mean sea surface temperature dataset
ds = xr.open_dataset("./sst.mnmean.nc")
# HTML representation of the dataset info. Remind yourself of the concept of coordinates
ds

## Extracting and Visualizing data
xarray comes with pandas and matplotlib capabilities. Hence you can extract data by indexing and visualize them, also adding the Cartopy mapping features. The matplotlib keywords specifying the type of plot are passed through the `DataArray.plot()` method. xarray makes a few educated guesses based on the shape of the data you have extracted. If the object is 1D, it shows a timeseries or a line `plot`; if its 2D shows a `pcolormesh` (changing the colormap depending on whether the data are all positive or positive and negative); in all other cases displays a histogram. 

### Indexing
Indexing is used to select specific elements from xarray files. Let’s select some data from the SST `DataArray`. We need to know that this DataArray has dimensions of time and two dimensional space (latitude and longitude): the first array index is time, the second is latitude, and so on.

You are probably already used to conventional ways of indexing an array. You would then use positional indexing:

In [ ]:
# select one variable and pick the first entry along all the axes
ds.sst[0,0,0]

In [ ]:
# Plot one timestep (the python convention includes all the other indexes)
ds.sst[0].plot()

In [ ]:
ds.sst[:,10,0].plot()

This method of handling arrays should be familiar to anyone who has worked with arrays in MATLAB or NumPy. Challenges with this approach: 
- *you need to know the order of the dimensions (time, lat, lon in this case, but it may change in different datasets)* 
- *it is not simple to associate an integer index position with something meaningful in our data (how do I know that index 10 of the second dimension is latitude 68S?)*

For example, we would have to write some function to map a specific date in the time dimension to its associated integer. **Note that even if you are using an array indexing, xarray still preserves the metadata and when you plot the extracted data you obtain an annotated figure!**

xarray lets us perform positional indexing using the coordinates instead of integers by using the methods 
- `isel` extracts data based on positional indexing along the labelled coordinates (you need to know the names, but not the order)
- `sel` extracts data using the coordinate values

They are equivalent to `iloc` and `loc` methods in `pandas`. 

In [ ]:
da = ds.sst
da.isel(lon=0,time=10,lat=0)

In [ ]:
da.isel(lat=60, lon=40).plot()

With method `da.isel()` you still need to know the correspondence between indexes and values. `da.sel()` allows you to do label-based indexing, with all the power of the pandas timeseries capabilities. In the following example we are also showing how you pass matplotlib keyword arguments (kwarg) through xarray plotting function:

In [ ]:
da.sel(lat=-32, lon=80).plot(figsize=(12,8),marker='o')

In [ ]:
da.sel(lat=50.0, lon=200.0, time="2020")

This method works if you match the exact coordinates of the data. If the coordinate *label* does not exist, and a `KeyError` is generated. 

xarray implements the keyword `method` to enable nearest neighbour (inexact) lookups by use of the methods `backfill` or `nearest`

In [ ]:
da.sel(lat=51.0, lon=200.0, time="2020")

In [ ]:
da.sel(lat=51., lon=200., method='nearest').plot()

The `slice` function can also be used, to select a range of coordinate values. Note that the method parameter `nearest` is not yet supported if any of the arguments to `.sel()` is a slice object

In [ ]:
# select a given period of time
da.sel(time=slice('2019-05', '2020-07')).plot()

<div class="alert alert-block alert-warning">
but wait, why do we see a histogram? What were you expecting?
    
<em> Think about the dimension of the extracted object... </em>
</div>

In [ ]:
# slicing can also be done along other axes
da.sel(time='2019-01',lat=-20,lon=slice(-50,80)).plot(marker='s')

<div class="alert alert-block alert-warning">
Where are the values with negative longitudes? 
    
<em> A quick look at the lon coordinate will give the answer... </em>
    
Why there are missing values?
</div>

In [ ]:
da.sel(time='2019-07',lat=slice(-20,-70),lon=slice(250,360)).plot()

### Mapping
This is very simple thanks to the integration with `cartopy`. If the axes on which you are plotting the object is a `GeoAxes` instance, the plot becomes a map!
Since xarray's default plotting functionality builds on matplotlib, we can seamlessly use cartopy to make nice maps:

1. Specify a `projection` for the plot when creating a new axis `axis`.
2. Explicitly ask xarray to plot to axis `axis` by passing the keyword argument `ax=axis`.
3. Specify the projection of the data using `transform` (`PlateCarree` here) in
   `.plot()`.

In [ ]:
axis = plt.axes(projection=ccrs.PlateCarree())
da.sel(time='2019-07').plot(ax=axis,transform=ccrs.PlateCarree(),
                           cbar_kwargs={'orientation': 'vertical', 'shrink': 0.6})
axis.set_extent([-110,10,-20,-70]) # now you can use all cartopy methods on the axis
axis.coastlines()  
gl = axis.gridlines(draw_labels=True)
gl.right_labels = False
gl.top_labels = False

In [ ]:
fig, axis = plt.subplots(1, 1,figsize=(10,10), subplot_kw=dict(projection=ccrs.Orthographic(0, -30)))

ds.sst.isel(time=1).plot(
    ax=axis,
    transform=ccrs.PlateCarree(),  # this is important since the data are on a mercator projection
    vmin=0., vmax=30., # these are matplotlib kwargs
    # some arguments passed to control the colorbar
    cbar_kwargs={"orientation": "horizontal", "shrink": 0.7},
    robust=True,
)
axis.coastlines()  # now you can use all cartopy methods on the axis
axis.gridlines()
# The parameter robust=True allows to visualize the data without the outliers, which may change your colorbar limits. 
# This will use the 2nd and 98th percentiles of the data to compute the color limits.

In [ ]:
#Let's visualize Antarctic sea-ice concentration data, provided on polar stereographic grid
#load the data and assess the metadata
ds_antarctic = xr.open_dataset("../shell-lesson-data/shell-lesson-data/exercise-data/data/asi-AMSR2-s6250-20260101-v5.4.nc")
ds_antarctic

In [ ]:
#extract SIC
SIC = ds_antarctic.z
SIC

In [ ]:
#map using the polar stereographic projection
fig, axis = plt.subplots(
    1, 1, figsize=(10, 10),
    subplot_kw=dict(projection=ccrs.SouthPolarStereo())
)
SIC.plot(
    ax=axis,
    vmin=0., vmax=100., # recall SIC ranges between 0 and 100 %
    cbar_kwargs={"orientation": "horizontal", "shrink": 0.7,"label":"Sea-ice concentration (%)"},
    robust=True,
    transform=ccrs.SouthPolarStereo()
)

axis.coastlines()
gl = axis.gridlines(draw_labels=True)
plt.show()

## Masking
Indexing methods on xarray objects generally return a subset of the original data. However, it is sometimes useful to select an object with the same shape as the original data, but with some elements masked. An example is selecting a given region, or all the gridpoints that have temperature larger than a given value.

To do this type of selection in xarray, we use the method `where()`:

In [ ]:
# tropical cyclones develop in regions where the surface temperature is greater than 26 degC
da.sel(time='2019-07').where(da>26.).plot()

In [ ]:
# which is better visualized with the mapping
fig,axis = plt.subplots(figsize=(15,7),subplot_kw=dict(projection=ccrs.PlateCarree()))
da.sel(time='2019-07').where(da>26.).plot(ax=axis,
                                          transform=ccrs.PlateCarree(),
                                         cbar_kwargs={'orientation': 'horizontal', 'shrink': 0.8})
axis.set_extent([-179,179,40,-40])
axis.coastlines()
gl=axis.gridlines(draw_labels=True)
gl.right_labels=False
gl.top_labels=False

Masking can also be used to extract a given region and to drop all the other points from the dataset. In this case, you use the keyword `drop=True`. This will return a dataset that is a portion of the original one. 

_Note: this may be an expensive operation and sometimes it's not efficient. Do it only if you need to reduce the memory footprint._

In [ ]:
import numpy as np
mask = np.logical_and((da.lon>0) & (da.lon<=30),(da.lat<-20) & (da.lat>=-36))
region = da.sel(time='2019-07').where(mask,drop=True)
print(region)
region.plot()

In [ ]:
#let's mask the sea-ice concentration data
sic_masked = SIC.where(SIC != 0)
sic_masked

In [ ]:
fig, axis = plt.subplots(
    1, 1, figsize=(10, 10),
    subplot_kw=dict(projection=ccrs.SouthPolarStereo())
)
sic_masked.plot(
    ax=axis,
    vmin=0., vmax=100., # recall SIC ranges between 0 and 100 %
    cbar_kwargs={"orientation": "horizontal", "shrink": 0.7,"label":"Sea-ice concentration (%)"},
    robust=True,
    transform=ccrs.SouthPolarStereo()
)

axis.add_feature(cfeature.OCEAN)
axis.add_feature(cfeature.LAND)

axis.coastlines()
gl = axis.gridlines(draw_labels=True)
plt.show()

## Going Further

- Xarray Documentation on Data Structures:
  http://xarray.pydata.org/en/latest/data-structures.html
- Xarray Documentation on Reading files and writing files:
  https://xarray.pydata.org/en/stable/io.html
- Xarrat Documentation on Indexing:
  http://xarray.pydata.org/en/stable/indexing.html
